In [ ]:
def main(datasources, start_date, end_date):
    """Pretrained v140 LOBFormer inference for BigQuant submission.

    Required weight file:
      models/v140_final_2020_2024_micro2_lobformer_rank90/weights.json
    """
    import base64
    import gc
    import json
    import os
    from pathlib import Path

    import dai
    import numpy as np
    import pandas as pd
    import structlog
    import torch
    import torch.nn as nn

    logger = structlog.get_logger()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    BASE_PRICE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
    LEVEL_PRICE_COLS = ["bid_price2", "ask_price2", "bid_price3", "ask_price3"]
    BASE_VOL_COLS = ["volume", "amount", "bid_volume1", "ask_volume1"]
    LEVEL_VOL_COLS = ["bid_volume2", "ask_volume2", "bid_volume3", "ask_volume3"]
    ORDER_COLS = [
        "deal_number",
        "bid_num_orders1",
        "ask_num_orders1",
        "bid_num_orders2",
        "ask_num_orders2",
        "bid_num_orders3",
        "ask_num_orders3",
    ]
    PRICE_COLS = BASE_PRICE_COLS + LEVEL_PRICE_COLS
    VOL_COLS = BASE_VOL_COLS + LEVEL_VOL_COLS
    MICRO_DERIVED_COLS = [
        "spread1",
        "spread2",
        "spread3",
        "imbalance1",
        "imbalance2",
        "imbalance3",
        "depth_imbalance3",
        "order_imbalance1",
        "order_imbalance3",
        "rel_open_mid",
        "rel_high_mid",
        "rel_low_mid",
        "rel_close_mid",
        "vwap_rel_mid",
        "log_deal_number",
    ]
    SEQUENCE_DERIVED_COLS = [
        "bar_pos",
        "time_sin",
        "time_cos",
        "ret1",
        "ret4",
        "ret16",
        "rv4",
        "rv16",
        "log_vol_chg1",
        "log_amount_chg1",
        "volume_z16",
        "amount_z16",
        "deal_z16",
        "spread1_chg1",
        "imbalance1_chg1",
        "spread1_mean4",
        "imbalance1_mean4",
        "depth_imbalance3_mean4",
    ]
    RICH_DERIVED_COLS = MICRO_DERIVED_COLS + SEQUENCE_DERIVED_COLS

    class StockLOBTransformer(nn.Module):
        def __init__(
            self,
            n_feat,
            seq_len,
            channels=256,
            d_model=256,
            nhead=8,
            nlayers=4,
            dim_ff=768,
            dropout=0.1,
        ):
            super().__init__()
            self.stem = nn.Sequential(
                nn.Conv1d(n_feat, channels, 3, padding=1),
                nn.BatchNorm1d(channels),
                nn.GELU(),
                nn.Conv1d(channels, channels, 3, padding=1),
                nn.BatchNorm1d(channels),
                nn.GELU(),
            )
            self.b3 = nn.Conv1d(channels, channels, 3, padding=1)
            self.b5 = nn.Conv1d(channels, channels, 5, padding=2)
            self.b9 = nn.Conv1d(channels, channels, 9, padding=4)
            self.mix = nn.Sequential(
                nn.Conv1d(channels * 3, d_model, 1),
                nn.BatchNorm1d(d_model),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
            layer = nn.TransformerEncoderLayer(
                d_model, nhead, dim_ff, dropout, batch_first=True, activation="gelu"
            )
            self.encoder = nn.TransformerEncoder(layer, nlayers)
            self.attn = nn.Linear(d_model, 1)
            self.head = nn.Sequential(
                nn.LayerNorm(d_model * 2),
                nn.Linear(d_model * 2, d_model),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model, 1),
            )

        def forward(self, x):
            h = self.stem(x.transpose(1, 2))
            h = self.mix(torch.cat([self.b3(h), self.b5(h), self.b9(h)], dim=1)).transpose(1, 2)
            h = self.encoder(h + self.pos[:, : h.shape[1]])
            w = torch.softmax(self.attn(h).squeeze(-1), dim=1).unsqueeze(-1)
            pooled = torch.cat([(h * w).sum(dim=1), h.mean(dim=1)], dim=1)
            return self.head(pooled).squeeze(-1)

    def decode_array(obj):
        arr = np.frombuffer(base64.b64decode(obj["data_b64"]), dtype=np.dtype(obj["dtype"]))
        return arr.reshape(obj["shape"]).copy()

    def decode_tensor(obj):
        return torch.from_numpy(decode_array(obj))

    def load_checkpoint(path):
        try:
            return torch.load(path, map_location=device, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=device)

    def weight_dir():
        rel = Path("models") / "v140_final_2020_2024_micro2_lobformer_rank90"
        candidates = []
        env_dir = os.environ.get("BQ_MODEL_DIR")
        if env_dir:
            candidates.append(Path(env_dir))
        candidates.extend([
            Path.cwd(),
            Path("/home/aiuser/work"),
            Path("/home/aiuser/work/我的克隆策略"),
            Path.cwd() / rel,
            Path("/home/aiuser/work") / rel,
            Path("/home/aiuser/work/v140_final_2020_2024_micro2_lobformer_rank90"),
            Path.cwd() / "v140_final_2020_2024_micro2_lobformer_rank90",
        ])
        for d in candidates:
            if (d / "weights.json").exists():
                return d
            if (d / "score15_l64.pt").exists() and (d / "score15_l32.pt").exists():
                return d
        raise FileNotFoundError(
            "missing v140 pretrained weights; put weights.json in the submission root "
            "or under models/v140_final_2020_2024_micro2_lobformer_rank90/"
        )

    def load_json_weights(wdir):
        path = wdir / "weights.json"
        if not path.exists():
            return None
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
        branches = {}
        for name, obj in payload["branches"].items():
            branches[name] = {
                "config": obj["config"],
                "feature_cols": obj["feature_cols"],
                "model_args": obj.get("model_args", {}),
                "stats": (
                    decode_array(obj["stats"]["mean"]).astype(np.float32),
                    decode_array(obj["stats"]["std"]).astype(np.float32),
                ),
                "state_dict": {k: decode_tensor(v) for k, v in obj["state_dict"].items()},
            }
        logger.info("loaded json weights", path=str(path), branches=list(branches))
        return branches

    def raw_cols_for_features(feature_cols):
        raw = [c for c in feature_cols if c not in set(RICH_DERIVED_COLS)]
        if any(c in feature_cols for c in RICH_DERIVED_COLS):
            for c in BASE_PRICE_COLS + LEVEL_PRICE_COLS + BASE_VOL_COLS + LEVEL_VOL_COLS + ORDER_COLS:
                if c not in raw:
                    raw.append(c)
        return raw

    def pool(table, sd, ed):
        df = dai.query(
            f"SELECT DISTINCT instrument FROM {table}",
            filters={"date": [sd, ed]},
        ).df()
        return df["instrument"].tolist()

    def add_features(df, feature_cols):
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"])
        for c in PRICE_COLS:
            if c in df.columns:
                df.loc[df[c] == -1, c] = np.nan
                df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
        for c in [c for c in VOL_COLS + ORDER_COLS if c in df.columns]:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")

        eps = 1e-6
        if any(c in feature_cols for c in MICRO_DERIVED_COLS):
            mid1 = (df["bid_price1"] + df["ask_price1"]) / 2.0
            for level in (1, 2, 3):
                bid = df[f"bid_price{level}"]
                ask = df[f"ask_price{level}"]
                bid_vol = df[f"bid_volume{level}"]
                ask_vol = df[f"ask_volume{level}"]
                df[f"spread{level}"] = (ask - bid) / (mid1.abs() + eps)
                df[f"imbalance{level}"] = (bid_vol - ask_vol) / (bid_vol + ask_vol + eps)
            bid_depth = df[["bid_volume1", "bid_volume2", "bid_volume3"]].sum(axis=1)
            ask_depth = df[["ask_volume1", "ask_volume2", "ask_volume3"]].sum(axis=1)
            df["depth_imbalance3"] = (bid_depth - ask_depth) / (bid_depth + ask_depth + eps)
            bid_orders1 = df["bid_num_orders1"]
            ask_orders1 = df["ask_num_orders1"]
            df["order_imbalance1"] = (bid_orders1 - ask_orders1) / (bid_orders1 + ask_orders1 + eps)
            bid_orders3 = df[["bid_num_orders1", "bid_num_orders2", "bid_num_orders3"]].sum(axis=1)
            ask_orders3 = df[["ask_num_orders1", "ask_num_orders2", "ask_num_orders3"]].sum(axis=1)
            df["order_imbalance3"] = (bid_orders3 - ask_orders3) / (bid_orders3 + ask_orders3 + eps)
            for col in ["open", "high", "low", "close"]:
                df[f"rel_{col}_mid"] = (df[col] - mid1) / (mid1.abs() + eps)
            vwap = df["amount"] / (df["volume"].replace(0, np.nan) + eps)
            df["vwap_rel_mid"] = (vwap - mid1) / (mid1.abs() + eps)
            df["log_deal_number"] = np.log1p(df["deal_number"].clip(lower=0))

        for c in [c for c in VOL_COLS + ORDER_COLS if c in df.columns]:
            df[c] = np.log1p(df[c].clip(lower=0))

        if any(c in feature_cols for c in SEQUENCE_DERIVED_COLS):
            minutes = df["date"].dt.hour * 60 + df["date"].dt.minute
            df["bar_pos"] = ((minutes - (9 * 60 + 45)).clip(lower=0, upper=240) / 240.0).astype("float32")
            phase = 2.0 * np.pi * df["bar_pos"]
            df["time_sin"] = np.sin(phase).astype("float32")
            df["time_cos"] = np.cos(phase).astype("float32")
            grp = df.groupby("instrument", sort=False)
            log_vol = np.log1p(df["volume"].clip(lower=0))
            log_amount = np.log1p(df["amount"].clip(lower=0))
            log_deal = np.log1p(df["deal_number"].clip(lower=0))
            df["ret1"] = grp["close"].transform(lambda s: np.log(s.replace(0, np.nan)).diff(1))
            df["ret4"] = grp["close"].transform(lambda s: np.log(s.replace(0, np.nan)).diff(4))
            df["ret16"] = grp["close"].transform(lambda s: np.log(s.replace(0, np.nan)).diff(16))
            df["rv4"] = grp["close"].transform(lambda s: np.log(s.replace(0, np.nan)).diff().rolling(4, min_periods=2).std())
            df["rv16"] = grp["close"].transform(lambda s: np.log(s.replace(0, np.nan)).diff().rolling(16, min_periods=4).std())
            df["log_vol_chg1"] = log_vol.groupby(df["instrument"]).diff(1)
            df["log_amount_chg1"] = log_amount.groupby(df["instrument"]).diff(1)
            vol_mean = log_vol.groupby(df["instrument"]).transform(lambda s: s.rolling(16, min_periods=4).mean())
            vol_std = log_vol.groupby(df["instrument"]).transform(lambda s: s.rolling(16, min_periods=4).std())
            amount_mean = log_amount.groupby(df["instrument"]).transform(lambda s: s.rolling(16, min_periods=4).mean())
            amount_std = log_amount.groupby(df["instrument"]).transform(lambda s: s.rolling(16, min_periods=4).std())
            deal_mean = log_deal.groupby(df["instrument"]).transform(lambda s: s.rolling(16, min_periods=4).mean())
            deal_std = log_deal.groupby(df["instrument"]).transform(lambda s: s.rolling(16, min_periods=4).std())
            df["volume_z16"] = (log_vol - vol_mean) / (vol_std + eps)
            df["amount_z16"] = (log_amount - amount_mean) / (amount_std + eps)
            df["deal_z16"] = (log_deal - deal_mean) / (deal_std + eps)
            df["spread1_chg1"] = grp["spread1"].diff(1)
            df["imbalance1_chg1"] = grp["imbalance1"].diff(1)
            df["spread1_mean4"] = grp["spread1"].transform(lambda s: s.rolling(4, min_periods=2).mean())
            df["imbalance1_mean4"] = grp["imbalance1"].transform(lambda s: s.rolling(4, min_periods=2).mean())
            df["depth_imbalance3_mean4"] = grp["depth_imbalance3"].transform(lambda s: s.rolling(4, min_periods=2).mean())

        df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return df

    def build_infer_dataset(table, sd, ed, instruments, seq_len, feature_cols, stats):
        buf = (pd.to_datetime(sd) - pd.Timedelta(days=20)).strftime("%Y-%m-%d")
        raw_cols = raw_cols_for_features(feature_cols)
        sql = f"SELECT date, instrument, {', '.join(raw_cols)} FROM {table} ORDER BY instrument, date"
        df = dai.query(sql, filters={"date": [buf, ed], "instrument": instruments}).df()
        if df.empty:
            return None, None
        df = add_features(df, feature_cols)
        sd_ts, ed_ts = pd.to_datetime(sd), pd.to_datetime(ed)

        wins, keys = [], []
        for ins, sub in df.groupby("instrument", sort=False):
            if len(sub) <= seq_len:
                continue
            feats = sub[feature_cols].to_numpy(np.float32)
            day = sub["date"].dt.normalize().to_numpy()
            close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))
            dates = day[close_pos]
            for p, d0 in zip(close_pos, dates):
                d = pd.Timestamp(d0)
                if p + 1 < seq_len or d < sd_ts or d > ed_ts:
                    continue
                wins.append(feats[p - seq_len + 1: p + 1])
                keys.append((d, ins))
        if not keys:
            return None, None

        X = np.stack(wins).astype(np.float32)
        mean, std = stats
        X = ((X - mean.astype(np.float32)) / (std.astype(np.float32) + 1e-6)).astype(np.float32)
        return X, pd.DataFrame(keys, columns=["date", "instrument"])

    def chunks(items, size):
        for i in range(0, len(items), size):
            yield items[i:i + size]

    def predict_branch(name, ck, table, instruments):
        cfg = ck["config"]
        feature_cols = list(ck["feature_cols"])
        state = ck["state_dict"]
        args = ck.get("model_args", {})
        channels = int(state["stem.0.weight"].shape[0])
        d_model = int(state["pos"].shape[-1])
        model = StockLOBTransformer(
            n_feat=len(feature_cols),
            seq_len=int(cfg["seq_len"]),
            channels=channels,
            d_model=d_model,
            nhead=int(args.get("nhead", 8)),
            nlayers=int(args.get("nlayers", 4)),
            dim_ff=int(args.get("dim_ff", 768)),
            dropout=float(args.get("dropout", 0.1)),
        ).to(device)
        model.load_state_dict(state)
        model.eval()
        batch = 2048 if device.type == "cuda" else 512
        inst_chunk = int(os.environ.get("BQ_INFER_INSTRUMENT_CHUNK", "192" if device.type == "cuda" else "80"))
        parts = []
        for ci, inst_part in enumerate(chunks(instruments, inst_chunk), start=1):
            X, idx = build_infer_dataset(
                table, start_date, end_date, inst_part, int(cfg["seq_len"]), feature_cols, ck["stats"]
            )
            if X is None or idx is None or idx.empty:
                logger.info("empty infer chunk skipped", name=name, chunk=ci, instruments=len(inst_part))
                continue
            xt = torch.from_numpy(X)
            preds = []
            with torch.no_grad():
                for i in range(0, len(idx), batch):
                    xb = xt[i:i + batch].to(device)
                    preds.append(model(xb).detach().cpu().numpy())
            part = idx.copy()
            part[name] = np.concatenate(preds).astype(np.float64)
            parts.append(part)
            logger.info("branch chunk predicted", name=name, chunk=ci,
                        instruments=len(inst_part), rows=len(part))
            del X, idx, xt, preds, part
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if not parts:
            raise RuntimeError(f"no inference rows for branch {name}: {table}, {start_date}~{end_date}")
        out = pd.concat(parts, ignore_index=True)
        out[name] = (out.groupby("date")[name].rank(method="average", pct=True) - 0.5).astype(np.float64)
        del model, parts
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        logger.info("branch predicted", name=name, rows=len(out), seq_len=int(cfg["seq_len"]))
        return out, float(cfg.get("weight", 1.0))

    wdir = weight_dir()
    json_weights = load_json_weights(wdir)
    infer_table = datasources.get("bar15m", "bigalpha_2026_stock_bar15m")
    instruments = pool(infer_table, start_date, end_date)
    logger.info("v140 pretrained inference start", weight_dir=str(wdir), table=infer_table,
                device=str(device), instruments=len(instruments))

    branches = []
    for name in ["score15_l64", "score15_l32"]:
        ck = json_weights[name] if json_weights is not None else load_checkpoint(wdir / f"{name}.pt")
        branches.append(predict_branch(name, ck, infer_table, instruments))

    merged = branches[0][0]
    for p, _ in branches[1:]:
        merged = pd.merge(merged, p, on=["date", "instrument"], how="outer")
    merged["score"] = 0.0
    weight_sum = 0.0
    for p, w in branches:
        col = [c for c in p.columns if c.startswith("score15_")][0]
        merged["score"] = merged["score"] + merged[col].fillna(0.0) * w
        weight_sum += w
    merged["score"] = merged["score"] / max(weight_sum, 1e-6)

    stk = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    result = (
        pd.merge(merged, stk, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
        .reset_index(drop=True)
    )
    result["score"] = (result.groupby("date")["score"].rank(method="average", pct=True) - 0.5).astype(np.float64)
    logger.info("v140 pretrained final score built", rows=len(result),
                days=result["date"].nunique(), instruments=result["instrument"].nunique())
    return result


def run_eval(start_date="2024-01-01 00:00:00", end_date="2024-12-31 23:59:59", show=True):
    """Notebook-side evaluation entry required by the BigQuant template."""
    from bigmodule import M

    datasources = {
        "bar15m": "bigalpha_2026_stock_bar15m",
        "bar5m": "bigalpha_2026_stock_bar5m",
        "bar1m": "bigalpha_2026_stock_bar1m",
    }
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    print(score_data.tail())
    print(score_data[["date", "instrument"]].nunique())
    result = M.bigalpha_eval._latest(factor_data=score_data, show=show)
    print(result)
    return result


if __name__ == "__main__":
    result = run_eval()
